# ST7 Project 2026

## Algorithm
1. **[LONG]** Forward solve: m_n → SEM3D → u_sim(x_ric, t) and u(x,t)
2. Misfit: r(t) = u_sim(t) - d_obs(t)
3. **[LONG]** Adjoint solve: r(T-t) → SEM3D backward → Λ(x,t)
4. Gradient: g_λ, g_μ from cross-correlation of ε[u] and ε[Λ]
5. CG direction (Fletcher-Reeves): p_n = -g_n + β_n · p_{n-1}
6. Line search (backtracking Armijo): find α_n
7. Update: m_{n+1} = m_n + α_n · p_n

## 0. Imports

In [ ]:
import numpy as np
import h5py
import os
import subprocess
import sys
import importlib
from pysem import parse_sem3d_traces

# pysem toolkit
from pysem.parse_sem3d_traces import ParseSEM3DH5Traces
from pysem.generate_h5_materials import write_h5
from util_func.sbatch_and_wait import sbatch_and_wait
from util_func.compute_misfit import compute_misfit
from util_func.read_stations_pos import read_stations_pos
from util_func.write_backward_spec import write_backward_spec_from_template
from util_func.write_misfit_files import write_time_reversed_residual_files

## 1. Paths and parameters

In [ ]:
os.mkdir("")

In [ ]:
# --- Paths ---
# WKDIR: working directory where SEM3D is launched
# must contain: input.spec, material.spec, stations.txt, gaussian_stf.txt, sem/mesh4spec.*.h5

FORWARD_PROBLEM_MESHER_SBATCH_PATH = "./sem3d_config_files/forward_problem_step1/MESHER.sbatch"
FORWARD_PROBLEM_SOLVER_SBATCH_PATH = "./sem3d_config_files/forward_problem_step1/SOLVER.sbatch"

TRACES_SIMULATED_FOLDER_PATH = ""
TRACES_OSSERVATED_PATH = "/usr/users/cea_seism/benede_gio/CEISM-Project/Uobs"

ADJOINT_DIR = "/path/to/adjoint_case"
MISFIT_DIR = os.path.join(ADJOINT_DIR, "adjoint_sources")
STATIONS_FILE_PATH = "/usr/users/cea_seism/tran_ngo/tutorials_SEM3D_DCE/tutorial2/stations.txt"
BACKWARD_INPUT_SPEC_TEMPLATE_PATH = "/path/to/input.spec"
BACKWARD_INPUT_SPEC_OUTPUT_PATH = ""


# D_OBS_DIR: directory containing observed data (tutorial2/prot/Protection_.../Capteurs/)

# --- Initial material parameters m_0 ---
# domain limits, discretization, initial gradient

# --- Optimization parameters ---
# N_ITER: maximum number of RTM iterations
# TOL: gradient convergence tolerance
# ALPHA_0: initial step size for line search
# ARMIJO_C: backtracking reduction constant
# ARMIJO_TAU: sufficient decrease factor

## 2. Load observed data d_obs

In [ ]:
obs_stream = ParseSEM3DH5Traces(
    wkdir=TRACES_OSSERVATED_PATH,
    format='h5',
    names=['Uobs'],
    variables=['Displ'],
    components=['x', 'y', 'z']
)

obs_monitor = obs_stream['Uobs']
obs_u = obs_monitor.data['Displ']

## 3. Initial material m_0

In [ ]:
# Generate initial material HDF5 files (e.g. homogeneous model)
# output: example_la.h5, example_mu.h5, example_ds.h5 in WKDIR

# m_la: 3D array of Lambda
# m_mu: 3D array of Mu
# m_ds: 3D array of Rho (fixed, not inverted)

## 4. Algorithm

In [ ]:
# CG variable initialization
# p_prev = 0  (previous direction)
# g_prev = 0  (previous gradient)
# J_prev = inf
N_ITER = 10 #to be modified

for n in range(N_ITER):

    # ── STEP 1 ────────────────────────────────────────
    sbatch_and_wait(FORWARD_PROBLEM_MESHER_SBATCH_PATH)
    sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)

    # ── STEP 2: MISFIT ────────────────────────────────────────────────────

    J, residual, t_sim, dt_sim = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, obs_u)
    
    # ── STEP 3 ────────────────────────────────────────

    time_reversed_residual = residual[::-1, :, :]

    stations = read_stations_pos(STATIONS_FILE_PATH)

    file_names = write_time_reversed_residual_files(time_reversed_residual, 
                                                    t_sim, 
                                                    OUTPUT_DIR=MISFIT_DIR)

    write_backward_spec_from_template(template_backward_spec_path=BACKWARD_INPUT_SPEC_TEMPLATE_PATH, 
                                      output_backward_spec_path = BACKWARD_INPUT_SPEC_OUTPUT_PATH, 
                                      stations = stations, 
                                      file_names = file_names, 
                                      misfit_rel_dir="adjoint_sources")

    sbatch_and_wait("SOLVER.sbatch")

    #input.spec
    #material

    # ── STEP 4: GRADIENT ─────────────────────────────────────────────────
    # Read edev and evol of u(x,t) from forward snapshots with ParseSEM3DSnapshots
    # Read edev and evol of Λ(x,t) from adjoint snapshots with ParseSEM3DSnapshots
    # Integrate in time (sum over all timesteps):
    #   g_la(x) = sum_t  evol[u](x,t) * evol[Λ](x,t)   (gradient w.r.t. Lambda)
    #   g_mu(x) = sum_t  edev[u](x,t) : edev[Λ](x,t)   (gradient w.r.t. Mu)
    # Add regularization term if present

    # Convergence check
    # if ||g|| < TOL: break

    # ── STEP 5: CONJUGATE GRADIENT DIRECTION (Fletcher-Reeves) ────────────
    # if n == 0:
    #     p_la = -g_la
    #     p_mu = -g_mu
    # else:
    #     beta = (||g||^2) / (||g_prev||^2)
    #     p_la = -g_la + beta * p_la_prev
    #     p_mu = -g_mu + beta * p_mu_prev

    # ── STEP 6: LINE SEARCH (backtracking Armijo) ─────────────────────────
    # alpha = ALPHA_0
    # while True:
    #     m_la_trial = m_la + alpha * p_la
    #     m_mu_trial = m_mu + alpha * p_mu
    #     Write m_trial to HDF5
    #     Launch forward solve with m_trial  ← (BLACK BOX)
    #     Compute J_trial
    #     if J_trial < J + ARMIJO_TAU * alpha * (g · p): break
    #     alpha = ARMIJO_C * alpha

    # ── STEP 7: UPDATE ─────────────────────────────────────────────────────
    # m_la = m_la + alpha * p_la
    # m_mu = m_mu + alpha * p_mu
    # Write new material HDF5 files
    # Save current state (for restart)
    # p_la_prev, p_mu_prev = p_la, p_mu
    # g_prev = g

NameError: name 'sim_u' is not defined